## Plotting the Gift-Eval Ablation Evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import os
from tsfm_lens.utils import apply_custom_style, normalize_by_seasonal_naive
from scipy.stats import gmean

#### Configuration

In [ ]:
model_name = "Chronos Bolt"

#### Presets

In [ ]:
model_to_dir_mapping = {
    "TimesFM 2.5": "google-timesfm-2.5-200m-pytorch",
    "Chronos Bolt": "amazon-chronos-bolt-base",
    "Chronos": "amazon-chronos-t5-base",
    "Toto": "Datadog-Toto-Open-Base-1.0_samples-20",
}

# Select the metric to analyze
SELECTED_METRIC = "eval_metrics/MASE[0.5]"
# SELECTED_METRIC = "eval_metrics/sMAPE[0.5]"

metric_pretty_name = "".join(c for c in SELECTED_METRIC.split("/")[-1] if c.isalpha())

print(f"Selectedmetric: {SELECTED_METRIC}")
print(f"Metric name: {metric_pretty_name}")


In [ ]:
apply_custom_style("../../config/plotting.yaml")
HOME_DIR = os.getenv("HOME")
root_dir = os.path.join(HOME_DIR, "tsfm-lens")  # type: ignore

save_figs = True
figs_save_dir = os.path.join(root_dir, "figs", f"gift-eval_{metric_pretty_name}", model_to_dir_mapping[model_name])
if not os.path.exists(figs_save_dir):
    os.makedirs(figs_save_dir)
if save_figs:
    print(f"Saving figures to: {figs_save_dir}")

## Load Data


In [ ]:
data_subdir_name = "heads1pp_rseed-42"
path_original_results = os.path.join(root_dir, "results", model_to_dir_mapping[model_name], data_subdir_name)
print(f"Loading original results from: {path_original_results}")
# list the files in the directory
print(os.listdir(path_original_results))

In [ ]:
original_results_df = pd.read_csv(os.path.join(path_original_results, "original_all_results.csv"))
# ablated_results_df = pd.read_csv(
#     os.path.join(
#         path_original_results,
#         "heads1pp_ablate_88_heads_za_heads_layers_7-8-9-10-11-12-13-mlps_layers_10-11_all_results.csv",
#         # "heads1pp_ablate_88_heads_za_heads_layers_7-8-9-10-11-12-13_all_results.csv",
#         # "heads1pp_ablate_164_heads_za_heads_layers_0-1-2-3-4-5-6-7-8-9-10-11-12-13-14-15-16-17-19-mlps_layers_10-11_all_results.csv"
#     )
# )
ablated_results_df = pd.read_csv(
    os.path.join(
        path_original_results,
        "heads1pp_ablate_74_heads_za_heads_layers_0-1-2-3-4-5-6-7-8-9-10-mlps_layers_3-4-5-6-7-8_all_results.csv",
        # "heads1pp_ablate_86_heads_za_heads_layers_0-1-2-3-4-5-6-7-8-9-10-mlps_layers_1-2-3-4-5-6-7-8-9_all_results.csv"
    )
)

## Normalize by Seasonal Naive Baseline


In [ ]:
# Configuration: Set to True to normalize metrics by seasonal naive baseline
NORMALIZE_BY_SEASONAL_NAIVE = True
is_normalized = False

print(f"Normalization by seasonal naive baseline: {NORMALIZE_BY_SEASONAL_NAIVE}")


In [ ]:
if NORMALIZE_BY_SEASONAL_NAIVE and not is_normalized:
    seasonal_naive_df = pd.read_csv(os.path.join(root_dir, "results", "seasonal_naive_baseline", "all_results.csv"))
    print(f"Loaded seasonal naive baseline: {len(seasonal_naive_df)} datasets. Normalizing...")
    original_results_df = normalize_by_seasonal_naive(original_results_df, seasonal_naive_df)
    ablated_results_df = normalize_by_seasonal_naive(ablated_results_df, seasonal_naive_df)
    is_normalized = True
else:
    print("Skipping normalization.")


### Display Available Metrics


In [ ]:
# Display available metrics
metric_columns = [col for col in original_results_df.columns if "eval_metrics" in col]
print("Available metrics:")
for i, metric in enumerate(metric_columns, 1):
    print(f"{i}. {metric}")


In [ ]:
geom_mean_original = gmean(original_results_df[SELECTED_METRIC])
geom_mean_ablated = gmean(ablated_results_df[SELECTED_METRIC])
print(f"Geom mean original: {geom_mean_original}")
print(f"Geom mean ablated: {geom_mean_ablated}")

percent_degradation = (geom_mean_ablated - geom_mean_original) / geom_mean_original * 100
print(f"Percent degradation: {percent_degradation:.2f}%")


## Identify Datasets close to Baseline

In [ ]:
# Get baseline and compute relative performance
# Create a baseline dataframe with just dataset and the selected metric
baseline = original_results_df[["dataset", SELECTED_METRIC]].copy()
baseline.columns = ["dataset", "baseline_value"]

# Merge ablated results with baseline
merged_data = ablated_results_df.merge(baseline, on="dataset")

# Compute relative performance (ablated / original)
merged_data["rel"] = merged_data[SELECTED_METRIC] / merged_data["baseline_value"]

In [ ]:
merged_data

In [ ]:
datasets_vulnerable = merged_data[merged_data["rel"] > 1.05]["dataset"].tolist()
print(f"{len(datasets_vulnerable)} Vulnerable datasets: {datasets_vulnerable}")

datasets_resilient = merged_data[(merged_data["rel"] > 1.0) & (merged_data["rel"] < 1.05)]["dataset"].tolist()
print(f"{len(datasets_resilient)} Resilient datasets: {datasets_resilient}")

datasets_improved = merged_data[merged_data["rel"] <= 1.0]["dataset"].tolist()
print(f"{len(datasets_improved)} Improved datasets: {datasets_improved}")